#Fake News Detection

## Data Preparation

In [3]:
import pandas as pd
import numpy as np

In [4]:
true = pd.read_csv('True.csv')
fake = pd.read_csv('Fake.csv')

In [5]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [6]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


### create new coloumn for label

In [7]:
true['label'] = 1
fake['label'] = 0

In [8]:
true.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [9]:
# merge two tables
news = pd.concat([true, fake]).reset_index(drop = True)

In [10]:
news.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [11]:
news.tail()

,title,text,subject,date,label
44893,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",0
44894,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",0
44895,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",0
44896,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",0
44897,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",0


In [12]:
news.isnull().sum()
# no null values found

,0
title,0
text,0
subject,0
date,0
label,0


In [13]:
# Drop unneeded coloumns
news = news.drop(['title', 'subject', 'date'], axis = 1)

In [14]:
news.head()

,text,label
0,WASHINGTON (Reuters) - The head of a conservat...,1
1,WASHINGTON (Reuters) - Transgender people will...,1
2,WASHINGTON (Reuters) - The special counsel inv...,1
3,WASHINGTON (Reuters) - Trump campaign adviser ...,1
4,SEATTLE/WASHINGTON (Reuters) - President Donal...,1


In [15]:
# Re shuffle the dataset to not make the model biased as the true news above and fake news below
news = news.sample(frac = 1)

In [16]:
news.head()

,text,label
29006,Republican governor Rick Scott of Florida had...,0
904,(Reuters) - The rollout of a tax-cut plan bein...,1
7607,WASHINGTON (Reuters) - Democratic presidential...,1
32577,A German artist is in a whole heap of trouble....,0
32261,This is truly where we re heading with freedom...,0


In [17]:
news = news.reset_index(drop = True)
# news = news.reset_index(inplace = True)

In [18]:
news.head()

,text,label
0,Republican governor Rick Scott of Florida had...,0
1,(Reuters) - The rollout of a tax-cut plan bein...,1
2,WASHINGTON (Reuters) - Democratic presidential...,1
3,A German artist is in a whole heap of trouble....,0
4,This is truly where we re heading with freedom...,0


## Feature Extraction

In [19]:
# make the text coloumn cleaner and remove all special characters or spaces
import re    # regular expression

In [20]:
def wordopt(text):
    text = text.lower()                     # convert into lower case
    text = re.sub(r'<.*?>', '', text)       # remove URLS
    text = re.sub(r'[^a-z0-9\s]', '', text) # remove special characters
    text = re.sub(r'\s+', ' ', text)        # remove extra spaces
    text = re.sub(r'[^\w\s]' , '' , text)   # remove puncitation
    text = re.sub(r'\d', '' , text)         # remove digits
    text = re.sub(r'\n', '', text)          # remove newline character
    text = text.strip()                     # remove leading and trailing spaces

    return text

In [21]:
news['text'] = news['text'].apply(wordopt);

In [22]:
news.head()

,text,label
0,republican governor rick scott of florida had ...,0
1,reuters the rollout of a taxcut plan being pro...,1
2,washington reuters democratic presidential can...,1
3,a german artist is in a whole heap of trouble ...,0
4,this is truly where we re heading with freedom...,0


In [23]:
x = news['text']
y = news['label']

In [24]:
x

,text
0,republican governor rick scott of florida had ...
1,reuters the rollout of a taxcut plan being pro...
2,washington reuters democratic presidential can...
3,a german artist is in a whole heap of trouble ...
4,this is truly where we re heading with freedom...
...,...
44893,in response to the establishment media s contr...
44894,the video below is one of the highlights from ...
44895,washington reuters an organization established...
44896,patrick henningsen st century wireany business...


In [25]:
y

,label
0,0
1,1
2,1
3,0
4,0
...,...
44893,0
44894,0
44895,1
44896,0


### Split the Dataset in to training and testing data on the percent of 30%

In [26]:
from sklearn.model_selection import train_test_split

In [27]:
x_train, x_text, y_train, y_test = train_test_split(x, y, test_size = 0.3)

In [28]:
x_train.shape

(31428,)

In [29]:
x_text.shape

(13470,)

In [30]:
# make the text coloumn more readable by machine, convert it to numerical coloumn
from sklearn.feature_extraction.text import TfidfVectorizer

In [31]:
vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_text)

In [32]:
xv_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6467153 stored elements and shape (31428, 177655)>

In [33]:
xv_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2716462 stored elements and shape (13470, 177655)>

## Create the ML models




### Logistic Regression

In [34]:
# start with logistic regression
from sklearn.linear_model import LogisticRegression

In [35]:
LR = LogisticRegression()
LR.fit(xv_train, y_train)

LogisticRegression()

In [36]:
# evaluate the prediction
pred_lr = LR.predict(xv_test)

In [37]:
LR.score(xv_test, y_test)

0.988641425389755

In [38]:
# call classification report
from sklearn.metrics import classification_report
print(classification_report(y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7044
           1       0.99      0.99      0.99      6426

    accuracy                           0.99     13470
   macro avg       0.99      0.99      0.99     13470
weighted avg       0.99      0.99      0.99     13470



### Decision Tree Classifier

In [39]:
# work with the decision tree classifier
from sklearn.tree import DecisionTreeClassifier

In [40]:
DTC = DecisionTreeClassifier()
DTC.fit(xv_train, y_train)

DecisionTreeClassifier()

In [41]:
pred_dtc = DTC.predict(xv_test)

In [42]:
DTC.score(xv_test, y_test)

0.995916852264291

In [43]:
print(classification_report(y_test, pred_dtc))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      7044
           1       1.00      1.00      1.00      6426

    accuracy                           1.00     13470
   macro avg       1.00      1.00      1.00     13470
weighted avg       1.00      1.00      1.00     13470



### Random forest Classifier

In [44]:
# work with the random forest classifier
from sklearn.ensemble import RandomForestClassifier

In [45]:
rfc = RandomForestClassifier(random_state = 0)
rfc.fit(xv_train, y_train)

RandomForestClassifier(random_state=0)

In [46]:
predict_rfc = rfc.predict(xv_test)

In [47]:
rfc.score(xv_test, y_test)

0.9872308834446919

In [48]:
print(classification_report(y_test, predict_rfc))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      7044
           1       0.98      0.99      0.99      6426

    accuracy                           0.99     13470
   macro avg       0.99      0.99      0.99     13470
weighted avg       0.99      0.99      0.99     13470



### Gradient Boosting Classifier

In [49]:
# work with the gradient boosting classifier
from sklearn.ensemble import GradientBoostingClassifier

In [50]:
gbc = GradientBoostingClassifier(random_state = 0)

In [51]:
gbc.fit(xv_train, y_train)

GradientBoostingClassifier(random_state=0)

In [52]:
predict_gbc = gbc.predict(xv_test)

In [53]:
gbc.score(xv_test, y_test)

0.9945805493689681

In [54]:
print(classification_report(y_test, predict_gbc))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      7044
           1       0.99      1.00      0.99      6426

    accuracy                           0.99     13470
   macro avg       0.99      0.99      0.99     13470
weighted avg       0.99      0.99      0.99     13470



## Create a Predicitive model to detect true or fake news

In [55]:
def output_lable(n):
    if n == 0:
      return "FAKE NEWS"
    elif n == 1:
      return "TRUE NEWS"

In [56]:
def manual_testing(news):
    testing_news = {"text":[news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(wordopt)
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)
    pred_lr = LR.predict(new_xv_test)
    pred_dtc = DTC.predict(new_xv_test)
    pred_gbc = gbc.predict(new_xv_test)
    pred_rfc = rfc.predict(new_xv_test)

    return "\n \nLR Prediction: {} \nDTC Prediction: {} \nGBC Prediction: {} \nRFC Prediction: {}".format(
        output_lable(pred_lr[0]),
        output_lable(pred_dtc[0]),
        output_lable(pred_gbc[0]),
        output_lable(pred_rfc[0])
    )

In [70]:
# 1
news_article = input("Enter news article: ")
print(news_article) # Russia has denied interfering in the U.S. election. U.S. Deputy Attorney General Rod Rosenstein

Enter news article: Russia has denied interfering in the U.S. election. U.S. Deputy Attorney General Rod Rosenstein 
Russia has denied interfering in the U.S. election. U.S. Deputy Attorney General Rod Rosenstein 


In [71]:
manual_testing(news_article)

'\n \nLR Prediction: TRUE NEWS \nDTC Prediction: FAKE NEWS \nGBC Prediction: FAKE NEWS \nRFC Prediction: FAKE NEWS'

In [68]:
# 2
news_article = input("Enter news article: ")
print(news_article) # Team USA’s women’s ice hockey team is set to face Canada in the Olympic gold medal game at the 2026 Winter Olympics in Milan-Cortina, with the U.S. dominating opponents throughout the tournament.

Enter news article: Team USA’s women’s ice hockey team is set to face Canada in the Olympic gold medal game at the 2026 Winter Olympics in Milan-Cortina, with the U.S. dominating opponents throughout the tournament.
Team USA’s women’s ice hockey team is set to face Canada in the Olympic gold medal game at the 2026 Winter Olympics in Milan-Cortina, with the U.S. dominating opponents throughout the tournament.


In [69]:
manual_testing(news_article)

'\n \nLR Prediction: FAKE NEWS \nDTC Prediction: FAKE NEWS \nGBC Prediction: FAKE NEWS \nRFC Prediction: FAKE NEWS'

In [65]:
# 3
news_article = input("Enter news article: ")
print(news_article) # U.S. national security officials have told President Trump that the military is prepared for potential strikes on Iran as soon as this weekend, although no final decision has been made yet.

Enter news article: U.S. national security officials have told President Trump that the military is prepared for potential strikes on Iran as soon as this weekend, although no final decision has been made yet.
U.S. national security officials have told President Trump that the military is prepared for potential strikes on Iran as soon as this weekend, although no final decision has been made yet.


In [66]:
manual_testing(news_article)

'\n \nLR Prediction: FAKE NEWS \nDTC Prediction: FAKE NEWS \nGBC Prediction: FAKE NEWS \nRFC Prediction: FAKE NEWS'